# Plot Validity: Jacobian & Hessian per event, per parameter

Loads precomputed results from `compute_taylor_scan.ipynb` and generates plots.
No GPU or simulation code needed here.

In [ ]:
import os, sys, pickle
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 13})


def load_results(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


def compute_validity_range(mean_true, mean_approx, rel_deltas, threshold):
    """Find asymmetric perturbation range where |approx - true| < threshold.
    Uses linear interpolation between grid points for exact crossing."""
    pct = rel_deltas * 100
    error = np.abs(mean_approx - mean_true)
    idx_nom = np.argmin(np.abs(rel_deltas))

    # Search left from nominal
    pct_lo = pct[0]
    for i in range(idx_nom - 1, -1, -1):
        if error[i] > threshold:
            # Crossing is between i (above) and i+1 (below)
            frac = (threshold - error[i + 1]) / (error[i] - error[i + 1] + 1e-30)
            pct_lo = pct[i + 1] + frac * (pct[i] - pct[i + 1])
            break

    # Search right from nominal
    pct_hi = pct[-1]
    for i in range(idx_nom + 1, len(pct)):
        if error[i] > threshold:
            # Crossing is between i-1 (below) and i (above)
            frac = (threshold - error[i - 1]) / (error[i] - error[i - 1] + 1e-30)
            pct_hi = pct[i - 1] + frac * (pct[i] - pct[i - 1])
            break

    return (float(pct_lo), float(pct_hi))


## Configuration

In [ ]:
RESULTS_DIR = 'results'
THRESHOLDS = [0.1, 0.5]  # ADC error thresholds for validity range


## Load results

In [ ]:
import glob

pkl_files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'taylor_scan_*.pkl')))
print(f'Found {len(pkl_files)} result files')

# Merge all results: combine events across input files
results = {}
for f in pkl_files:
    input_id = os.path.basename(f).replace('taylor_scan_', '').replace('.pkl', '')
    data = load_results(f)
    for param_name, param_data in data.items():
        if param_name not in results:
            results[param_name] = {}
        for eid, eid_data in param_data.items():
            # Prefix event ID with input_id to avoid collisions
            key = f'{input_id}_{eid}'
            results[param_name][key] = eid_data

param_names = list(results.keys())
event_ids = sorted(next(iter(results.values())).keys()) if results else []
print(f'Parameters: {param_names}')
print(f'Total events: {len(event_ids)}')


## Per-parameter: mean ADC curves

For each parameter, one subplot per event showing true / Jacobian / Hessian.

In [ ]:
for param_name in param_names:
    param_results = results[param_name]
    eids = sorted(param_results.keys())[0:24]
    n_events = len(eids)
    ncols = 6
    nrows = (n_events + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(2*ncols, 2*nrows), sharex=True)
    axes_flat = axes.flatten()

    for i, eid in enumerate(eids):
        ax = axes_flat[i]
        sr = param_results[eid]
        pct = sr['rel_deltas'] * 100
        n_pix = sr['n_active']
        H_val = sr['H_scalar']

        ax.plot(pct, sr['mean_true'], 'ko-', ms=2, lw=1.2, label='True')
        ax.plot(pct, sr['mean_jac'],  'C0^--', ms=2, lw=1, label='Jac')
        ax.plot(pct, sr['mean_hess'], 'C1s--', ms=2, lw=1, label='Hess')
        ax.axvline(0, color='gray', ls=':', lw=0.5)
        ax.set_title(f'Evt {eid} ({n_pix}px, H={H_val:.1f})', fontsize=8)
        if i == 0:
            ax.legend(fontsize=6, loc='best')

    for j in range(i+1, len(axes_flat)):
        axes_flat[j].set_visible(False)
    for ax in axes[-1, :]:
        if ax.get_visible():
            ax.set_xlabel(f'$\\Delta${param_name} [%]', fontsize=8)
    for ax in axes[:, 0]:
        ax.set_ylabel('Mean ADC', fontsize=8)

    fig.suptitle(f'Parameter: {param_name}', fontsize=14)
    fig.tight_layout()
    os.makedirs('plots', exist_ok=True)
    fig.savefig(f'plots/validity_{param_name}_per_event.pdf', bbox_inches='tight')
    plt.show()

## Relative error of Taylor approximations

For each perturbation level, compute the per-event relative error:

$$\varepsilon_{\rm rel} = \frac{|\Delta\text{ADC}_{\rm approx} - \Delta\text{ADC}_{\rm true}|}{|\Delta\text{ADC}_{\rm true}|}$$

where $\Delta\text{ADC} = \langle\text{ADC}\rangle_{\rm perturbed} - \langle\text{ADC}\rangle_{\rm nominal}$.

Plot the 68th and 90th percentiles of this relative error across events
as a function of perturbation %.

In [ ]:
PERCENTILES = [68, 90]
PERCENTILE_STYLES = {68: '-', 90: '--'}

for param_name in param_names:
    param_results = results[param_name]
    eids = sorted(param_results.keys())

    # Get the perturbation grid from any event
    rel_deltas = next(iter(param_results.values()))['rel_deltas']
    pct = rel_deltas * 100
    idx_nom = np.argmin(np.abs(rel_deltas))

    # Compute per-event relative errors at each perturbation
    rel_err_jac = []
    rel_err_hess = []

    for eid in eids:
        sr = param_results[eid]
        delta_true = sr['mean_true'] - sr['mean_true'][idx_nom]
        delta_jac  = sr['mean_jac']  - sr['mean_jac'][idx_nom]
        delta_hess = sr['mean_hess'] - sr['mean_hess'][idx_nom]

        denom = np.abs(delta_true)
        denom_safe = np.where(denom > 1e-10, denom, np.nan)

        err_j = np.abs(delta_jac - delta_true) / denom_safe
        err_h = np.abs(delta_hess - delta_true) / denom_safe
        # At nominal point, relative error is exactly 0
        err_j[idx_nom] = 0.0
        err_h[idx_nom] = 0.0
        rel_err_jac.append(err_j)
        rel_err_hess.append(err_h)

    rel_err_jac = np.array(rel_err_jac)
    rel_err_hess = np.array(rel_err_hess)

    fig, ax = plt.subplots(figsize=(8, 5))

    for p in PERCENTILES:
        ls = PERCENTILE_STYLES[p]
        jac_pct  = np.nanpercentile(rel_err_jac * 100, p, axis=0)
        hess_pct = np.nanpercentile(rel_err_hess * 100, p, axis=0)

        ax.plot(pct, jac_pct,  color='C0', ls=ls, lw=1.5, label=f'Jacobian ({p}th %ile)')
        ax.plot(pct, hess_pct, color='C1', ls=ls, lw=1.5, label=f'Hessian ({p}th %ile)')

    ax.set_xlabel(f'$\\Delta${param_name} / {param_name}$_{{nom}}$ [%]')
    ax.set_ylabel('Relative error [%]')
    ax.set_title(f'Relative error of Taylor approximations: {param_name}')
    ax.axvline(0, color='gray', ls=':', lw=0.5)
    ax.set_ylim(bottom=0, top=100)
    ax.legend(fontsize=9)

    fig.tight_layout()
    os.makedirs('plots', exist_ok=True)
    fig.savefig(f'plots/relative_error_{param_name}.pdf', bbox_inches='tight')
    plt.show()
